In [ ]:
!nvidia-smi

Mon Apr 27 19:17:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!unzip -q /content/drive/MyDrive/MVProject/sidewalk_hazard_dataset.zip -d /content/MVProject

!echo "Top-level dataset folder:"
!ls /content/MVProject

Top-level dataset folder:
annotations  class_map.json  images


In [ ]:
from pathlib import Path

candidates = list(Path("/content/MVProject").rglob("train.json"))
print("Found train.json files:")
for p in candidates:
    print(p)

assert len(candidates) > 0, "Could not find train.json. Check your dataset zip path/structure."

# Dataset root is the parent of annotations/
DATA_ROOT = str(candidates[0].parent.parent) + "/"
print("DATA_ROOT =", DATA_ROOT)

print("\nDataset root contents:")
!find "$DATA_ROOT" -maxdepth 2 -type d | sort

Found train.json files:
/content/MVProject/annotations/train.json
DATA_ROOT = /content/MVProject/

Dataset root contents:
/content/MVProject/
/content/MVProject/annotations
/content/MVProject/images
/content/MVProject/images/train
/content/MVProject/images/val


In [ ]:
import json
from pathlib import Path
from collections import Counter

data_root = Path(DATA_ROOT)

for split in ["train", "val"]:
    ann_path = data_root / "annotations" / f"{split}.json"
    img_dir = data_root / "images" / split

    assert ann_path.exists(), f"Missing {ann_path}"
    assert img_dir.exists(), f"Missing {img_dir}"

    data = json.load(open(ann_path))
    image_ids = {img["id"] for img in data["images"]}
    bad_annotations = [ann for ann in data["annotations"] if ann["image_id"] not in image_ids]

    missing_images = []
    for img in data["images"][:1000]:  # sample first 1000 for speed
        if not (img_dir / img["file_name"]).exists():
            missing_images.append(img["file_name"])

    cat_lookup = {cat["id"]: cat["name"] for cat in data["categories"]}
    class_counts = Counter(cat_lookup[ann["category_id"]] for ann in data["annotations"])

    print(f"--- {split.upper()} ---")
    print("Images:", len(data["images"]))
    print("Annotations:", len(data["annotations"]))
    print("Bad annotations:", len(bad_annotations))
    print("Missing sampled images:", len(missing_images))
    print("Categories:", [cat["name"] for cat in data["categories"]])
    print("Top class counts:", class_counts.most_common(20))
    print()

    assert len(bad_annotations) == 0, f"{split} has bad annotations"
    assert len(missing_images) == 0, f"{split} has missing image files in sampled check"

--- TRAIN ---
Images: 11636
Annotations: 145501
Bad annotations: 0
Missing sampled images: 0
Categories: ['person', 'bicycle', 'car', 'traffic_cone', 'pothole', 'open_manhole', 'pole', 'sign', 'trash_can', 'roadblock', 'tree', 'animal', 'fire_hydrant', 'crosswalk', 'tactile_paving', 'surface_damage']
Top class counts: [('person', 28442), ('car', 26498), ('pole', 25026), ('tree', 18210), ('bicycle', 15836), ('traffic_cone', 11645), ('crosswalk', 6849), ('roadblock', 3569), ('sign', 2674), ('trash_can', 2285), ('tactile_paving', 1909), ('fire_hydrant', 1101), ('animal', 823), ('pothole', 339), ('surface_damage', 164), ('open_manhole', 131)]

--- VAL ---
Images: 2909
Annotations: 35393
Bad annotations: 0
Missing sampled images: 0
Categories: ['person', 'bicycle', 'car', 'traffic_cone', 'pothole', 'open_manhole', 'pole', 'sign', 'trash_can', 'roadblock', 'tree', 'animal', 'fire_hydrant', 'crosswalk', 'tactile_paving', 'surface_damage']
Top class counts: [('person', 6803), ('car', 6409), ('

In [ ]:
%cd /content

# Install micromamba if not already present
!if [ ! -f /content/bin/micromamba ]; then \
  curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba; \
fi

!/content/bin/micromamba --version

# Create clean Python 3.10 environment
!/content/bin/micromamba create -y -n openmmlab python=3.10 -c conda-forge

# Show Python version inside the env
!/content/bin/micromamba run -n openmmlab python --version

/content
bin/micromamba
2.5.0
[+] 0.0s
[+] 0.1s
[+] 0.2s
conda-forge/linux-64   1%
conda-forge/noarch     8%[+] 0.3s
conda-forge/linux-64  10%
conda-forge/noarch    29%[+] 0.4s
conda-forge/linux-64  17%
conda-forge/noarch    43%[+] 0.5s
conda-forge/linux-64  22%
conda-forge/noarch    52%[+] 0.6s
conda-forge/linux-64  29%
conda-forge/noarch    66%[+] 0.7s
conda-forge/linux-64  35%
conda-forge/noarch    78%[+] 0.8s
conda-forge/linux-64  42%
conda-forge/noarch    93%conda-forge/noarch                                
[+] 0.9s
conda-forge/linux-64  46%[+] 1.0s
conda-forge/linux-64  46%[+] 1.1s
conda-forge/linux-64  46%[+] 1.2s
conda-forge/linux-64  46%[+] 1.3s
conda-forge/linux-64  46%[+] 1.4s
conda-forge/linux-64  46%[+] 1.5s
conda-forge/linux-64  46%[+] 1.6s
conda-forge/linux-64  46%[+] 1.7s
conda-forge/linux-64  46%[+] 1.8s
conda-forge/linux-64  48%[+] 1.9s
conda-forge/linux-64  48%[+] 2.0s
conda-forge/linux-64  48%[+] 2.1s
conda-forge/linux-64  48%[+] 2.2s
conda-forge/linux-64  48%[+] 2

In [ ]:
# Core build tools
!/content/bin/micromamba run -n openmmlab pip install -U pip setuptools wheel

# PyTorch CUDA 11.8 wheels
!/content/bin/micromamba run -n openmmlab pip install \
  torch==2.1.0 torchvision==0.16.0 torchaudio==2.1.0 \
  --index-url https://download.pytorch.org/whl/cu118

# OpenMMLab packages
!/content/bin/micromamba run -n openmmlab pip install -U openmim

# MMEngine
!/content/bin/micromamba run -n openmmlab mim install "mmengine"

# Full MMCV with compiled ops
!/content/bin/micromamba run -n openmmlab mim install "mmcv==2.1.0"

# MMDetection
!/content/bin/micromamba run -n openmmlab pip install "mmdet==3.3.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 47.9 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 26.0.1
    Uninstalling pip-26.0.1:
      Successfully uninstalled pip-26.0.1
Looking in indexes: https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 GB 8.9 MB/s  0:00:54
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 22.1 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 76.4 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.2/89.2 MB 48.8 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 104.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 102.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 147.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 148.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 32.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
#Install other depencies found during testing
%%bash
/content/bin/micromamba run -n openmmlab pip install "numpy<2" "matplotlib<3.9"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 118.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 122.8 MB/s  0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6
  Attempting uninstall: matplotlib
    Found existing installation: matplotlib 3.10.9
    Uninstalling matplotlib-3.10.9:
      Successfully uninstalled matplotlib-3.10.9



ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.


In [ ]:
# Fallback direct wheel install for CUDA 11.8 + Torch 2.1.0.
# Only run if the previous MMCV install failed.

# !/content/bin/micromamba run -n openmmlab pip uninstall -y mmcv mmcv-lite
# !/content/bin/micromamba run -n openmmlab pip install "mmcv==2.1.0" \
#   -f https://download.openmmlab.com/mmcv/dist/cu118/torch2.1.0/index.html

In [ ]:
!/content/bin/micromamba run -n openmmlab pip list | grep -E "torch|mmcv|mmengine|mmdet|openmim"

mmcv                   2.1.0
mmdet                  3.3.0
mmengine               0.10.7
openmim                0.3.9
torch                  2.1.0+cu118
torchaudio             2.1.0+cu118
torchvision            0.16.0+cu118


In [ ]:
%%bash
MPLBACKEND=Agg /content/bin/micromamba run -n openmmlab python - <<'PY'
import numpy
import torch
import matplotlib
import mmcv
import mmcv._ext
import mmdet

print("NumPy:", numpy.__version__)
print("Torch:", torch.__version__)
print("Matplotlib backend:", matplotlib.get_backend())
print("MMCV:", mmcv.__version__)
print("MMDetection:", mmdet.__version__)
print("Ready")
PY

NumPy: 1.26.4
Torch: 2.1.0+cu118
Matplotlib backend: agg
MMCV: 2.1.0
MMDetection: 3.3.0
Ready


In [ ]:
%cd /content
!rm -rf mmdetection
!git clone --branch v3.3.0 https://github.com/open-mmlab/mmdetection.git
%cd /content/mmdetection

!ls configs/rtmdet

/content
Cloning into 'mmdetection'...
remote: Enumerating objects: 38023, done.
remote: Total 38023 (delta 0), reused 0 (delta 0), pack-reused 38023 (from 1)
Receiving objects: 100% (38023/38023), 63.18 MiB | 23.69 MiB/s, done.
Resolving deltas: 100% (26221/26221), done.
Note: switching to '44ebd17b145c2372c4b700bfb9cb20dbd28ab64a'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

/content/mmdetection
classification			    rtmdet_l_convnext_b_4xb32-100e_coco.py
metafile.yml			    rtmdet_l_swin_b_4xb32-100e_coco.py
README.md			

In [ ]:
%cd /content/mmdetection

!mkdir -p checkpoints

!wget -q https://download.openmmlab.com/mmdetection/v3.0/rtmdet/rtmdet_tiny_8xb32-300e_coco/rtmdet_tiny_8xb32-300e_coco_20220902_112414-78e30dcc.pth \
  -O checkpoints/rtmdet_tiny_coco.pth

!ls -lh checkpoints/rtmdet_tiny_coco.pth

/content/mmdetection
-rw-r--r-- 1 root root 55M Sep 25  2022 checkpoints/rtmdet_tiny_coco.pth


In [ ]:
%%writefile /content/make_rtmdet_config.py
from mmengine.config import Config

BASE_CONFIG = "/content/mmdetection/configs/rtmdet/rtmdet_tiny_8xb32-300e_coco.py"
OUT_CONFIG = "/content/mmdetection/configs/rtmdet/rtmdet_tiny_sidewalk_hazards.py"

DATA_ROOT = "/content/MVProject/"
WORK_DIR = "/content/drive/MyDrive/MVProject/rtmdet_tiny_work_dir"
CHECKPOINT = "/content/drive/MyDrive/MVProject/checkpoints/rtmdet_tiny_coco.pth"

CLASSES = (
    "person",
    "bicycle",
    "car",
    "traffic_cone",
    "pothole",
    "open_manhole",
    "pole",
    "sign",
    "trash_can",
    "roadblock",
    "tree",
    "animal",
    "fire_hydrant",
    "crosswalk",
    "tactile_paving",
    "surface_damage",
)

cfg = Config.fromfile(BASE_CONFIG)

cfg.dataset_type = "CocoDataset"
cfg.data_root = DATA_ROOT
cfg.metainfo = dict(classes=CLASSES)

# Data paths
cfg.train_dataloader.dataset.type = "CocoDataset"
cfg.train_dataloader.dataset.data_root = DATA_ROOT
cfg.train_dataloader.dataset.ann_file = "annotations/train.json"
cfg.train_dataloader.dataset.data_prefix = dict(img="images/train/")
cfg.train_dataloader.dataset.metainfo = cfg.metainfo

cfg.val_dataloader.dataset.type = "CocoDataset"
cfg.val_dataloader.dataset.data_root = DATA_ROOT
cfg.val_dataloader.dataset.ann_file = "annotations/val.json"
cfg.val_dataloader.dataset.data_prefix = dict(img="images/val/")
cfg.val_dataloader.dataset.metainfo = cfg.metainfo

cfg.test_dataloader = cfg.val_dataloader

cfg.val_evaluator.ann_file = DATA_ROOT + "annotations/val.json"
cfg.test_evaluator = cfg.val_evaluator

# Custom class count
cfg.model.bbox_head.num_classes = len(CLASSES)

# COCO-pretrained weights
cfg.load_from = CHECKPOINT

# Colab-friendly settings
cfg.train_dataloader.batch_size = 2
cfg.train_dataloader.num_workers = 1
cfg.val_dataloader.batch_size = 2
cfg.val_dataloader.num_workers = 1
cfg.test_dataloader.batch_size = 2
cfg.test_dataloader.num_workers = 1

# Smoke test first
cfg.train_cfg.max_epochs = 10
cfg.train_cfg.val_interval = 1

# Lower LR for fine-tuning
cfg.optim_wrapper.optimizer.lr = 1e-4

# Mixed precision
cfg.optim_wrapper.type = "AmpOptimWrapper"

# Checkpointing
cfg.work_dir = WORK_DIR
cfg.default_hooks.checkpoint.interval = 1
cfg.default_hooks.checkpoint.max_keep_ckpts = 5
cfg.default_hooks.logger.interval = 50

# Reduce memory by using 512x512 instead of default 640x640
# RTMDet config usually has train_pipeline/test_pipeline available.
for pipeline in [cfg.train_pipeline, cfg.test_pipeline]:
    for step in pipeline:
        if "scale" in step:
            step["scale"] = (512, 512)

cfg.dump(OUT_CONFIG)

print("Wrote config:", OUT_CONFIG)
print("Classes:", CLASSES)
print("num_classes:", len(CLASSES))
print("data_root:", DATA_ROOT)
print("work_dir:", WORK_DIR)

Overwriting /content/make_rtmdet_config.py


In [ ]:
!/content/bin/micromamba run -n openmmlab python /content/make_rtmdet_config.py

!sed -n '1,220p' /content/mmdetection/configs/rtmdet/rtmdet_tiny_sidewalk_hazards.py

Wrote config: /content/mmdetection/configs/rtmdet/rtmdet_tiny_sidewalk_hazards.py
Classes: ('person', 'bicycle', 'car', 'traffic_cone', 'pothole', 'open_manhole', 'pole', 'sign', 'trash_can', 'roadblock', 'tree', 'animal', 'fire_hydrant', 'crosswalk', 'tactile_paving', 'surface_damage')
num_classes: 16
data_root: /content/MVProject/
work_dir: /content/drive/MyDrive/MVProject/rtmdet_tiny_work_dir
auto_scale_lr = dict(base_batch_size=16, enable=False)
backend_args = None
base_lr = 0.004
checkpoint = 'https://download.openmmlab.com/mmdetection/v3.0/rtmdet/cspnext_rsb_pretrain/cspnext-tiny_imagenet_600e.pth'
custom_hooks = [
    dict(
        ema_type='ExpMomentumEMA',
        momentum=0.0002,
        priority=49,
        type='EMAHook',
        update_buffers=True),
    dict(
        switch_epoch=280,
        switch_pipeline=[
            dict(backend_args=None, type='LoadImageFromFile'),
            dict(type='LoadAnnotations', with_bbox=True),
            dict(
                keep_rati

In [ ]:
%%bash
MPLBACKEND=Agg /content/bin/micromamba run -n openmmlab python - <<'PY'
import numpy
import torch
import matplotlib
import mmcv
import mmcv._ext
import mmdet

print("NumPy:", numpy.__version__)
print("Torch:", torch.__version__)
print("Matplotlib backend:", matplotlib.get_backend())
print("MMCV:", mmcv.__version__)
print("MMDetection:", mmdet.__version__)
print("Ready")
PY

NumPy: 1.26.4
Torch: 2.1.0+cu118
Matplotlib backend: agg
MMCV: 2.1.0
MMDetection: 3.3.0
Ready


In [ ]:
!cd /content/mmdetection && MPLBACKEND=Agg PYTHONUNBUFFERED=1 /content/bin/micromamba run -n openmmlab python -u tools/train.py configs/rtmdet/rtmdet_tiny_sidewalk_hazards.py 2>&1 | tee /content/drive/MyDrive/MVProject/rtmdet_training_log.txt

04/25 20:59:48 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.10.20 | packaged by conda-forge | (main, Mar  5 2026, 16:42:22) [GCC 14.3.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 533326196
    GPU 0: Tesla T4
    CUDA_HOME: /usr/local/cuda
    NVCC: Cuda compilation tools, release 12.8, V12.8.93
    GCC: gcc (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0
    PyTorch: 2.1.0+cu118
    PyTorch compiling details: PyTorch built with:
  - GCC 9.3
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2022.2-Product Build 20220804 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v3.1.1 (Git Hash 64f6bcbcbab628e96f33a62c3e975f8535a7bde4)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: AVX512
  - CUDA Runtime 11.8
  - NVCC architecture flags: -gencode;arch=compu

In [ ]:
!cd /content/mmdetection && MPLBACKEND=Agg PYTHONUNBUFFERED=1 /content/bin/micromamba run -n openmmlab python -u tools/train.py configs/rtmdet/rtmdet_tiny_sidewalk_hazards.py --resume /content/drive/MyDrive/MVProject/rtmdet_tiny_work_dir/epoch_8.pth

04/27 19:34:04 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.10.20 | packaged by conda-forge | (main, Mar  5 2026, 16:42:22) [GCC 14.3.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 48570771
    GPU 0: Tesla T4
    CUDA_HOME: /usr/local/cuda
    NVCC: Cuda compilation tools, release 12.8, V12.8.93
    GCC: gcc (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0
    PyTorch: 2.1.0+cu118
    PyTorch compiling details: PyTorch built with:
  - GCC 9.3
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2022.2-Product Build 20220804 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v3.1.1 (Git Hash 64f6bcbcbab628e96f33a62c3e975f8535a7bde4)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: AVX512
  - CUDA Runtime 11.8
  - NVCC architecture flags: -gencode;arch=comput

In [ ]:
%cd /content/mmdetection

CHECKPOINT_PATH = "/content/drive/MyDrive/MVProject/rtmdet_tiny_work_dir/epoch_3.pth"

!/content/bin/micromamba run -n openmmlab python tools/test.py \
  configs/rtmdet/rtmdet_tiny_sidewalk_hazards.py \
  "$CHECKPOINT_PATH"

In [ ]:
from pathlib import Path

val_images = sorted(Path(DATA_ROOT).joinpath("images", "val").glob("*"))
assert len(val_images) > 0, "No validation images found."

TEST_IMAGE = str(val_images[0])
print("TEST_IMAGE =", TEST_IMAGE)

In [ ]:
%cd /content/mmdetection

CHECKPOINT_PATH = "/content/drive/MyDrive/MVProject/rtmdet_tiny_work_dir/epoch_3.pth"

!rm -rf /content/rtmdet_demo_outputs

!/content/bin/micromamba run -n openmmlab python demo/image_demo.py \
  "$TEST_IMAGE" \
  configs/rtmdet/rtmdet_tiny_sidewalk_hazards.py \
  --weights "$CHECKPOINT_PATH" \
  --out-dir /content/rtmdet_demo_outputs

In [ ]:
from IPython.display import Image, display
import glob

outputs = glob.glob("/content/rtmdet_demo_outputs/*")
print(outputs)

if outputs:
    display(Image(outputs[0]))
else:
    print("No output image found.")